# 08 — Diagnóstico de baselines (M2: patógeno)

**Objetivo.** Determinar, con una corrida de bajo costo (tres configuraciones, una semilla, presupuesto y datos
reducidos), si el modelo propuesto es competitivo frente a baselines de una sola entrada, antes de invertir cómputo
en la comparación completa.

**Tecnología.** TensorFlow/Keras con EfficientNet y MobileNetV2 preentrenados en ImageNet, Albumentations y
scikit-learn. Ejecución en Google Colab con GPU; persistencia de resultados y caché en Google Drive.

**Metodología.** Se comparan el modelo propuesto (doble entrada, hoja aislada por segmentación M_seg, normalización
Shades-of-Gray y promediado exponencial de pesos) y dos baselines de una sola entrada, con partición y semilla fijas.
El promediado exponencial emplea calentamiento del factor de decaimiento para evitar el sesgo de inicialización en
entrenamientos cortos. Métrica principal: F1 macro sobre el conjunto de prueba independiente.

In [ ]:
!pip install -q tensorflow albumentations scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_SUBDIR = 'glycine_vision_baselines'
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive') / DRIVE_SUBDIR
OUT.mkdir(parents=True, exist_ok=True)
assert str(OUT).startswith('/content/drive') and OUT.exists(), 'Drive no montado: el avance se perdería al desconectar.'
print('Persistencia (resultados + caché) en Drive:', OUT)

In [ ]:
import json, math, pickle, re, time
from pathlib import Path
import numpy as np
import cv2
import tensorflow as tf
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence
from sklearn.metrics import f1_score, accuracy_score

IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_P1 = 6
EPOCHS_P2 = 10
LR_P1 = 1e-3
LR_P2 = 1e-5
EMA_DECAY = 0.9999
LABEL_SMOOTH = 0.05
SEEDS = [42]
NUM_CLASSES = 5
SUBSAMPLE_PER_CLASS = 400
LOADER = dict(workers=4, use_multiprocessing=False, max_queue_size=10)
DATA = Path('/content/splits/clasificacion_patogeno')
RAW_PATH = OUT / 'raw_results_m2.csv'
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Presupuesto: fase1=', EPOCHS_P1, 'fase2=', EPOCHS_P2, '| lote=', BATCH, '| semillas=', SEEDS, '| submuestra/clase=', SUBSAMPLE_PER_CLASS)

In [ ]:
_gpus = tf.config.list_physical_devices('GPU')
if not _gpus:
    print('=' * 64)
    print('SIN GPU DETECTADA. En CPU el entrenamiento es ~20x más lento,')
    print('lo que provoca cortes de sesión antes de terminar una configuración.')
    print('Runtime -> Cambiar tipo de entorno de ejecución -> GPU (T4) -> reejecutar.')
    print('=' * 64)
else:
    print('GPU activa:', _gpus[0].name)

In [ ]:
import shutil, zipfile

DRIVE_ZIP = OUT / 'splits.zip'
if not DATA.exists():
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError(
            'No hay dataset local ni ' + str(DRIVE_ZIP) + '. Genera ./splits con el notebook 01 y, una sola vez, '
            'guarda el comprimido en Drive:  shutil.make_archive(str(OUT / \'splits\'), \'zip\', \'/content\', \'splits\')')
    print('Copiando dataset comprimido desde Drive (una vez por sesión)...')
    _t0 = time.time()
    _local_zip = Path('/content/splits.zip')
    shutil.copy(DRIVE_ZIP, _local_zip)
    with zipfile.ZipFile(_local_zip) as _z:
        _z.extractall('/content')
    print('Dataset local listo en %.1f min.' % ((time.time() - _t0) / 60))

TEST_DIR = Path('/content/splits/test/clasificacion_patogeno')
if not TEST_DIR.exists():
    TEST_DIR = DATA / 'val'

SEG_KERAS = next((p for p in [OUT / 'model_seg.keras', Path('/content/outputs/model_seg.keras'), Path('/content/model_seg.keras')] if p.exists()), None)
SEG_TFLITE = next((p for p in [OUT / 'model_seg.tflite', Path('/content/outputs/model_seg.tflite'), Path('/content/model_seg.tflite')] if p.exists()), None)
assert SEG_KERAS or SEG_TFLITE, 'Falta model_seg.keras o model_seg.tflite (colócalo en la carpeta de Drive junto a splits.zip).'
print('Datos:', DATA, '| Test:', TEST_DIR)
print('Segmentador:', SEG_KERAS or SEG_TFLITE)

In [ ]:
_MASK_CACHE_PATH = OUT / 'mseg_mask_cache.pkl'
_MASK_CACHE = pickle.loads(_MASK_CACHE_PATH.read_bytes()) if _MASK_CACHE_PATH.exists() else {}
_IMG_CACHE = {}
_SEG = None
print('Máscaras en caché:', len(_MASK_CACHE))


def _load_seg():
    global _SEG
    if _SEG is None:
        if SEG_KERAS is not None:
            _SEG = ('keras', tf.keras.models.load_model(SEG_KERAS, compile=False))
        else:
            interp = tf.lite.Interpreter(model_path=str(SEG_TFLITE)); interp.allocate_tensors()
            _SEG = ('tflite', interp)
    return _SEG


def seg_leaf_masks(batch01):
    kind, obj = _load_seg()
    if kind == 'keras':
        probs = obj.predict(batch01, verbose=0, batch_size=len(batch01))
        return (np.argmax(probs, -1) == 1).astype(np.uint8)
    inp = obj.get_input_details()[0]; out = obj.get_output_details()[0]
    res = []
    for img in batch01:
        obj.set_tensor(inp['index'], img[np.newaxis].astype(inp['dtype']))
        obj.invoke()
        a = obj.get_tensor(out['index'])[0]
        res.append((np.argmax(a, -1) == 1).astype(np.uint8) if a.ndim == 3 else (a > 0).astype(np.uint8))
    return np.stack(res)


def cached_read(fp, size):
    arr = _IMG_CACHE.get(fp)
    if arr is None:
        for _ in range(3):
            try:
                arr = np.array(Image.open(fp).convert('RGB').resize(size))
                break
            except (OSError, IOError):
                time.sleep(0.3)
        if arr is not None:
            _IMG_CACHE[fp] = arr
    return arr.copy() if arr is not None else None


def chromatic_normalize(img_rgb):
    x = img_rgb.astype(np.float32)
    illum = np.power(np.mean(np.power(x, 6), axis=(0, 1)), 1.0 / 6.0)
    scale = np.clip(illum.mean() / (illum + 1e-6), 0.6, 1.6)
    return np.clip(x * scale, 0, 255).astype(np.uint8)


def save_mask_cache():
    _MASK_CACHE_PATH.write_bytes(pickle.dumps(_MASK_CACHE))


def precompute_masks(paths, batch_size=32):
    pending = [fp for fp in paths if fp not in _MASK_CACHE]
    print(f'Máscaras M_seg por calcular: {len(pending)} de {len(paths)}.')
    if not pending:
        return
    for start in range(0, len(pending), batch_size):
        chunk = pending[start:start + batch_size]
        imgs, valid_fps = [], []
        for fp in chunk:
            img = cached_read(fp, IMG_SIZE)
            if img is None:
                continue
            small = chromatic_normalize(cv2.resize(img, (256, 256)))
            imgs.append(small.astype(np.float32) / 255.0)
            valid_fps.append(fp)
        if not imgs:
            continue
        leaves = seg_leaf_masks(np.stack(imgs))
        for fp, leaf in zip(valid_fps, leaves):
            _MASK_CACHE[fp] = cv2.resize(leaf, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
        if start % (batch_size * 20) == 0:
            save_mask_cache()
            print(f'  {start + len(chunk)}/{len(pending)}')
    save_mask_cache()
    print('Caché de máscaras guardada.')

In [ ]:
class ConfigSequence(Sequence):
    def __init__(self, directory, cfg, batch_size, augment, shuffle, aug=None, subsample_per_class=None, **kwargs):
        super().__init__(**kwargs)
        self.cfg = cfg
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.aug = aug
        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            files = []
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                files += list((Path(directory) / cls).glob(ext))
            if subsample_per_class is not None and len(files) > subsample_per_class:
                rng = np.random.default_rng(42)
                files = list(rng.choice(files, size=subsample_per_class, replace=False))
            self.samples += [(str(fp), i) for fp in files]
        self.n = len(self.samples)
        self.labels = np.array([s[1] for s in self.samples])
        if shuffle:
            np.random.shuffle(self.samples)
            self.labels = np.array([s[1] for s in self.samples])

    def all_paths(self):
        return [fp for fp, _ in self.samples]

    def __len__(self):
        return max(1, math.ceil(self.n / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size:(idx + 1) * self.batch_size]
        orig, leaf, labels = [], [], []
        for fp, cls in batch:
            img = cached_read(fp, IMG_SIZE)
            if img is None:
                continue
            mask = _MASK_CACHE.get(fp) if self.cfg['isolate'] == 'mseg' else None
            if self.augment and self.aug is not None:
                if mask is not None:
                    aug = self.aug(image=img, mask=mask); img, mask = aug['image'], aug['mask']
                else:
                    img = self.aug(image=img)['image']
            base = chromatic_normalize(img) if self.cfg['shades'] else img.astype(np.uint8)
            orig.append(base.astype(np.float32))
            if self.cfg['dual']:
                iso = base.copy()
                if mask is not None:
                    iso[mask == 0] = 0
                leaf.append(iso.astype(np.float32))
            labels.append(cls)
        if not orig:
            z = np.zeros((1, *IMG_SIZE, 3), np.float32)
            y = np.zeros((1, NUM_CLASSES), np.float32)
            return ((z, z) if self.cfg['dual'] else z), y
        X_o = np.stack(orig)
        Y = np.eye(NUM_CLASSES, dtype=np.float32)[labels]
        if self.cfg['dual']:
            return (X_o, np.stack(leaf)), Y
        return X_o, Y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.samples)
            self.labels = np.array([s[1] for s in self.samples])


class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.9999):
        super().__init__(); self.decay = decay; self.ema = None; self.step = 0
    def on_train_begin(self, logs=None):
        self.ema = [w.numpy().copy() for w in self.model.trainable_weights]
        self.step = 0
    def on_train_batch_end(self, batch, logs=None):
        self.step += 1
        effective = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        for i, w in enumerate(self.model.trainable_weights):
            self.ema[i] = effective * self.ema[i] + (1.0 - effective) * w.numpy()
    def on_train_end(self, logs=None):
        for w, e in zip(self.model.trainable_weights, self.ema):
            w.assign(e)


def _backbone_fn(name):
    if name == 'mobilenet':
        return tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input
    if name == 'b1':
        return tf.keras.applications.EfficientNetB1, tf.keras.applications.efficientnet.preprocess_input
    return tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input


def build_model(cfg, size=IMG_SIZE, l2=5e-5):
    Base, prep = _backbone_fn(cfg['backbone'])
    base = Base(weights='imagenet', include_top=False, input_shape=(*size, 3))
    base.trainable = False
    inp_o = tf.keras.layers.Input((*size, 3), name='original')
    fo = tf.keras.layers.GlobalAveragePooling2D()(base(tf.keras.layers.Lambda(prep)(inp_o)))
    if cfg['dual']:
        inp_l = tf.keras.layers.Input((*size, 3), name='hoja_aislada')
        fl = tf.keras.layers.GlobalAveragePooling2D()(base(tf.keras.layers.Lambda(prep)(inp_l)))
        x = tf.keras.layers.Concatenate()([fo, fl])
        inputs = [inp_o, inp_l]
    else:
        x = fo
        inputs = inp_o
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2))(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    safe_name = re.sub(r'[^A-Za-z0-9]', '_', cfg['name'])[:40]
    model = tf.keras.Model(inputs, out, name=safe_name)
    model._backbone = base
    return model


def set_finetune(model, last_n=80):
    base = model._backbone
    base.trainable = True
    for layer in base.layers[:-last_n]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

In [ ]:
TRAIN_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.Affine(translate_percent=0.08, scale=(0.88, 1.15), rotate=0, p=0.4),
])
LOSS = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)


def evaluate_on_test(model, cfg):
    test_gen = ConfigSequence(TEST_DIR, cfg, BATCH, augment=False, shuffle=False, subsample_per_class=None, **LOADER)
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)[:test_gen.n]
    y_true = test_gen.labels[:len(y_pred)]
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average='macro')


def train_and_eval(cfg, seed):
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed); np.random.seed(seed)
    train_gen = ConfigSequence(DATA / 'train', cfg, BATCH, augment=True, shuffle=True, aug=TRAIN_AUG, subsample_per_class=SUBSAMPLE_PER_CLASS, **LOADER)
    val_gen = ConfigSequence(DATA / 'val', cfg, BATCH, augment=False, shuffle=False, subsample_per_class=SUBSAMPLE_PER_CLASS, **LOADER)
    counts = np.bincount(train_gen.labels, minlength=NUM_CLASSES)
    cw = {i: float(counts.sum() / (NUM_CLASSES * max(1, counts[i]))) for i in range(NUM_CLASSES)}
    model = build_model(cfg)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_P1), loss=LOSS, metrics=['accuracy'])
    es1 = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=3, restore_best_weights=True)
    model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P1, callbacks=[es1], class_weight=cw, verbose=0)
    set_finetune(model, last_n=80)
    steps = max(1, len(train_gen)) * EPOCHS_P2
    lr = tf.keras.optimizers.schedules.CosineDecay(LR_P2, steps, alpha=0.01)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=LOSS, metrics=['accuracy'])
    cbs = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True)]
    if cfg['ema']:
        cbs.append(EMACallback(EMA_DECAY))
    model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P2, callbacks=cbs, class_weight=cw, verbose=0)
    return evaluate_on_test(model, cfg)

In [ ]:
PROPOSED = dict(name='Propuesto (dual + M_seg + SoG + EMA)', backbone='b0', dual=True, isolate='mseg', shades=True, ema=True)
BEST_BASELINE = dict(name='EfficientNetB0 (single, cruda)', backbone='b0', dual=False, isolate='none', shades=False, ema=False)

BASELINES_ONLY = [
    dict(name='MobileNetV2 (single, cruda)', backbone='mobilenet', dual=False, isolate='none', shades=False, ema=False),
    BEST_BASELINE,
]

ABLATION = [
    PROPOSED,
    {**PROPOSED, 'name': 'sin doble entrada (solo original)', 'dual': False},
    {**PROPOSED, 'name': 'sin hoja aislada M_seg (2a entrada cruda)', 'isolate': 'none'},
    {**PROPOSED, 'name': 'sin Shades-of-Gray', 'shades': False},
    {**PROPOSED, 'name': 'sin EMA', 'ema': False},
]

DIAG_CONFIGS = BASELINES_ONLY + [PROPOSED]
ALL_CONFIGS = list({cfg['name']: cfg for cfg in BASELINES_ONLY + ABLATION}.values())

In [ ]:
import pandas as pd

COLUMNS = ['config_name', 'seed', 'accuracy', 'f1_macro']


def load_raw():
    if RAW_PATH.exists():
        return pd.read_csv(RAW_PATH).drop_duplicates(subset=['config_name', 'seed'], keep='last')
    return pd.DataFrame(columns=COLUMNS)


def append_raw(row):
    df = pd.concat([load_raw(), pd.DataFrame([row])], ignore_index=True)
    df = df.drop_duplicates(subset=['config_name', 'seed'], keep='last')
    df.to_csv(RAW_PATH, index=False)


def run_all(configs, seeds):
    done = load_raw()
    for cfg in configs:
        for seed in seeds:
            if ((done['config_name'] == cfg['name']) & (done['seed'] == int(seed))).any():
                print(f"  [ya guardado] {cfg['name']:42s} seed={seed}")
                continue
            t0 = time.time()
            acc, f1 = train_and_eval(cfg, int(seed))
            append_raw({'config_name': cfg['name'], 'seed': int(seed), 'accuracy': acc, 'f1_macro': f1})
            done = load_raw()
            print(f"  {cfg['name']:42s} seed={seed}  acc={acc:.3f}  f1={f1:.3f}  ({(time.time()-t0)/60:.1f} min, guardado)")


def summarize(config_names):
    df = load_raw()
    rows = []
    for name in config_names:
        sub = df[df['config_name'] == name]
        rows.append({
            'Configuración': name,
            'Exactitud (media)': round(float(sub['accuracy'].mean()), 3),
            'Exactitud (std)': round(float(sub['accuracy'].std(ddof=0)), 3),
            'F1 macro (media)': round(float(sub['f1_macro'].mean()), 3),
            'F1 macro (std)': round(float(sub['f1_macro'].std(ddof=0)), 3),
            'N': len(sub),
        })
    return pd.DataFrame(rows)

In [ ]:
SMOKE_TEST = True
if SMOKE_TEST:
    tf.keras.backend.clear_session(); tf.random.set_seed(42); np.random.seed(42)
    _g = ConfigSequence(DATA / 'train', BEST_BASELINE, BATCH, augment=False, shuffle=True, subsample_per_class=40, **LOADER)
    _v = ConfigSequence(DATA / 'val', BEST_BASELINE, BATCH, augment=False, shuffle=False, subsample_per_class=40, **LOADER)
    _m = build_model(BEST_BASELINE); _m.compile(optimizer='adam', loss=LOSS, metrics=['accuracy'])
    _t0 = time.time(); _m.fit(_g, validation_data=_v, epochs=1, verbose=0); _dt = time.time() - _t0
    (OUT / 'smoke_test.txt').write_text('ok')
    assert (OUT / 'smoke_test.txt').exists()
    print(f'Prueba de guardado OK ({OUT}). Una época mínima en {_dt:.0f}s.')
    if _dt > 90:
        print('AVISO: tardó demasiado para una época mínima. Verifica que el entorno tenga GPU.')

In [ ]:
RESET_RAW = True
if RESET_RAW and RAW_PATH.exists():
    RAW_PATH.replace(RAW_PATH.with_name('raw_results_m2_previo.csv'))
    print('Resultados previos archivados.')

In [ ]:
_scan_cfg = {'isolate': 'mseg'}
all_paths = set()
for _split_dir in (DATA / 'train', DATA / 'val', TEST_DIR):
    gen = ConfigSequence(_split_dir, _scan_cfg, BATCH, augment=False, shuffle=False, subsample_per_class=None)
    all_paths.update(gen.all_paths())
precompute_masks(sorted(all_paths), batch_size=BATCH)

In [ ]:
run_all(DIAG_CONFIGS, SEEDS)
df = summarize([c['name'] for c in DIAG_CONFIGS])
df.to_csv(OUT / 'diagnostico_m2.csv', index=False)
df

In [ ]:
prop = df.loc[df['Configuración'] == PROPOSED['name'], 'F1 macro (media)'].iloc[0]
best_base = df.loc[df['Configuración'].isin([c['name'] for c in BASELINES_ONLY]), 'F1 macro (media)'].max()
print(f'Propuesto: {prop:.3f}   |   Mejor baseline: {best_base:.3f}')
if prop >= best_base:
    print('El propuesto iguala o supera a los baselines. Continuar con el notebook 09.')
elif prop > 0.905:
    print('El propuesto se recuperó pero queda por debajo. Verificar al presupuesto completo en el notebook 10.')
else:
    print('El propuesto no se recuperó. Revisar el diagnóstico antes de invertir más cómputo.')